## Exasol Prod connection

In [1]:
import pyexasol
import configparser

#Location of the ini file
config = configparser.ConfigParser()
config.read('C:\\Users\\USER\\.spyder-py3\\ExasolPROD.ini')
dsn=config['exasolPROD']['dsn']
user=config['exasolPROD']['user']
pwd=config['exasolPROD']['pwd']
schema=config['exasolPROD']['schema']

# Exasol connection
connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)

### Query

In [3]:
#sql_query =  input("Please enter the query: ")

In [2]:
sql_query = """--- 2019 DATA
-- JF
WITH HOTELS_2020_AFTER_LOCKDOWN AS(
SELECT 
DISTINCT A.HOTEL_ID
FROM
(
SELECT distinct 
BKG.BOOKING_ID,
BKG.HOTEL_ID,
CO.HOTEL_COUNTRY_NAME,
CON.CONTINENT_NAME,
HOC.HOTEL_CATEGORY_SHORT_DESC,
HOC.HOTEL_CATEGORY_NUMBER,
--SD.SOURCING_DESTINATION,
HO.HOTEL_COUNTRY_ID,
HO.HOTEL_CITY_ID,
HO.HOTEL_CATEGORY_ID,
BKG.HOTEL_ARRIVAL_DATE,
BKG.BOOKING_DATE,
BKG.AMOUNT_TURNOVER,
BKG.NUMBER_BOOKED_ROOMNIGHTS ,
CASE WHEN BOOKING_ORIGIN_COUNTRY_ID = HO.HOTEL_COUNTRY_ID THEN 1 END AS 'DOMESTIC',
CASE WHEN  BOOKING_ORIGIN_COUNTRY_ID <> HO.HOTEL_COUNTRY_ID THEN 1 END AS 'INTERNATIONAL',
CASE WHEN BKG.BOOKING_STATUS_ID IN (0,1) THEN 'SUCCESFFUL_BKG'
ELSE 'CANCELLED_BKG' END AS 'BKG_STATUS'
FROM DWHBIL.FAK_BOOKING BKG 
LEFT JOIN DWHBIL.V_LKP_HOTEL HO ON HO.HOTEL_ID = BKG.HOTEL_ID
LEFT JOIN DWHBIL.FAK_RATEGAIN GA ON GA.HOTEL_ID = HO.HOTEL_ID
LEFT JOIN DWHBIL.V_LKP_HOTEL_CATEGORY HOC ON HOC.HOTEL_CATEGORY_ID = HO.HOTEL_CATEGORY_ID
LEFT JOIN DWHBIL.V_LKP_HOTEL_COUNTRY CO ON CO.HOTEL_COUNTRY_ID = HO.HOTEL_COUNTRY_ID
LEFT JOIN DWHBIL.V_LKP_CONTINENT CON ON CON.CONTINENT_ID = CO.HOTEL_CONTINENT_ID
JOIN DWHBIL.REL_CITY_TO_SOURCING_DESTINATION SD ON SD.HOTEL_CITY_ID = HO.HOTEL_CITY_ID
WHERE 
 BKG.MICE_ID  = -1 AND BKG.BOOKING_SOURCE_ID <> 742 --AND BKG.BOOKING_STATUS_ID IN (0,1)
AND (YEAR(BKG.BOOKING_DATE) IN (2020) AND MONTH(BKG.BOOKING_DATE) IN (1,2))
AND (YEAR(BKG.HOTEL_ARRIVAL_DATE) IN (2020) AND MONTH(BKG.HOTEL_ARRIVAL_DATE) IN (1,2))--AND LOCAL.HOTE_ADR BETWEEN 20 AND 500
AND HOC.HOTEL_CATEGORY_NUMBER IN (3,4,5) 
AND BKG.NUMBER_BOOKED_ROOMNIGHTS > 0  AND BKG.AMOUNT_TURNOVER/BKG.NUMBER_BOOKED_ROOMNIGHTS BETWEEN 10 AND 500
) A 
)
, HOTELS_2019_JF AS(

SELECT distinct 
BKG.BOOKING_ID,
BKG.HOTEL_ID,
CO.HOTEL_COUNTRY_NAME,
CON.CONTINENT_NAME,
HOC.HOTEL_CATEGORY_SHORT_DESC,
HOC.HOTEL_CATEGORY_NUMBER,
--SD.SOURCING_DESTINATION,
HO.HOTEL_COUNTRY_ID,
HO.HOTEL_CITY_ID,
HO.HOTEL_CATEGORY_ID,
(BKG.HOTEL_ARRIVAL_DATE) ,
(BKG.BOOKING_DATE) ,
(BKG.AMOUNT_TURNOVER) ,
(BKG.NUMBER_BOOKED_ROOMNIGHTS),
CASE WHEN BOOKING_ORIGIN_COUNTRY_ID = HO.HOTEL_COUNTRY_ID THEN 1 END AS 'DOMESTIC',
CASE WHEN  BOOKING_ORIGIN_COUNTRY_ID <> HO.HOTEL_COUNTRY_ID THEN 1 END AS 'INTERNATIONAL',
CASE WHEN  BOOKING_ORIGIN_COUNTRY_ID IS NULL THEN 1 END AS 'UNKNOWNTRAV',
CASE WHEN BKG.BOOKING_STATUS_ID IN (0,1) THEN 'SUCCESFFUL_BKG'
ELSE 'CANCELLED_BKG' END AS 'BKG_STATUS'
FROM DWHBIL.FAK_BOOKING BKG 
LEFT JOIN DWHBIL.V_LKP_HOTEL HO ON HO.HOTEL_ID = BKG.HOTEL_ID
LEFT JOIN DWHBIL.FAK_RATEGAIN GA ON GA.HOTEL_ID = HO.HOTEL_ID
LEFT JOIN DWHBIL.V_LKP_HOTEL_CATEGORY HOC ON HOC.HOTEL_CATEGORY_ID = HO.HOTEL_CATEGORY_ID
LEFT JOIN DWHBIL.V_LKP_HOTEL_COUNTRY CO ON CO.HOTEL_COUNTRY_ID = HO.HOTEL_COUNTRY_ID
LEFT JOIN DWHBIL.V_LKP_CONTINENT CON ON CON.CONTINENT_ID = CO.HOTEL_CONTINENT_ID
JOIN DWHBIL.REL_CITY_TO_SOURCING_DESTINATION SD ON SD.HOTEL_CITY_ID = HO.HOTEL_CITY_ID
WHERE 
 BKG.MICE_ID  = -1 AND BKG.BOOKING_SOURCE_ID <> 742 --AND BKG.BOOKING_STATUS_ID IN (0,1)
AND (
((YEAR(BKG.BOOKING_DATE) IN (2019) AND MONTH(BKG.BOOKING_DATE) IN (1,2,3))AND (YEAR(BKG.HOTEL_ARRIVAL_DATE) IN (2019) AND MONTH(BKG.HOTEL_ARRIVAL_DATE) IN (1,2,3)))
OR 
((YEAR(BKG.BOOKING_DATE) IN (2019) AND MONTH(BKG.BOOKING_DATE) IN (4,5,6)) AND (YEAR(BKG.HOTEL_ARRIVAL_DATE) IN (2019) AND MONTH(BKG.HOTEL_ARRIVAL_DATE) IN (4,5,6)))
OR 
((YEAR(BKG.BOOKING_DATE) IN (2020) AND MONTH(BKG.BOOKING_DATE) IN (1,2,3)) AND (YEAR(BKG.HOTEL_ARRIVAL_DATE) IN (2020) AND MONTH(BKG.HOTEL_ARRIVAL_DATE) IN (1,2,3)))
OR 
((YEAR(BKG.BOOKING_DATE) IN (2020) AND MONTH(BKG.BOOKING_DATE) IN (4,5,6)) AND (YEAR(BKG.HOTEL_ARRIVAL_DATE) IN (2020) AND MONTH(BKG.HOTEL_ARRIVAL_DATE) IN (4,5,6)))
)

--AND LOCAL.HOTE_ADR BETWEEN 20 AND 500
AND HOC.HOTEL_CATEGORY_NUMBER IN (3,4,5) 
AND BKG.NUMBER_BOOKED_ROOMNIGHTS > 0  AND BKG.AMOUNT_TURNOVER / BKG.NUMBER_BOOKED_ROOMNIGHTS BETWEEN 10 AND 500
AND HO.HOTEL_COUNTRY_ID <> 29
AND HO.HOTEL_ID IN (SELECT distinct HOTEL_ID FROM HOTELS_2020_AFTER_LOCKDOWN)

)
SELECT DISTINCT 
COUNT(BKG.BOOKING_ID) AS BOOKING_COUNT,
BKG.HOTEL_ID,
BKG.HOTEL_COUNTRY_NAME,
BKG.CONTINENT_NAME,
BKG.HOTEL_CATEGORY_SHORT_DESC,
BKG.HOTEL_CATEGORY_NUMBER,
--SD.SOURCING_DESTINATION,
BKG.HOTEL_COUNTRY_ID,
BKG.HOTEL_CITY_ID,
BKG.HOTEL_CATEGORY_ID,
YEAR(BKG.HOTEL_ARRIVAL_DATE) AS ARR_YEAR ,
MONTH(BKG.HOTEL_ARRIVAL_DATE) AS ARR_MONTH,
YEAR(BKG.BOOKING_DATE) AS BKG_YEAR,
MONTH(BKG.BOOKING_DATE) AS BKG_MONTH,
SUM(BKG.AMOUNT_TURNOVER) AS TOVER,
SUM(BKG.NUMBER_BOOKED_ROOMNIGHTS) AS RNS,
SUM(DOMESTIC) AS DOM_TRAV,
SUM(INTERNATIONAL) AS INT_TRAV,
SUM(UNKNOWNTRAV) AS UNKNOWN_TRAV,
BKG.BKG_STATUS,
LOCAL.TOVER / LOCAL.RNS AS ADR,
MPD_19.MPD_FOR_DEVELOPMENT_YEAR AS MPD_2019,
MPD_20.MPD_FOR_DEVELOPMENT_YEAR AS MPD_2020
FROM HOTELS_2019_JF BKG 
JOIN DWHBIL.REF_MARKET_PRICE_DEVELOPMENT_PER_MONTH MPD_19 ON MPD_19.HOTEL_CITY_ID  = BKG.HOTEL_CITY_ID AND MPD_19.MPD_BASE_YEAR = 2018 AND MPD_19.DEVELOPMENT_YEAR = 2019
JOIN DWHBIL.REF_MARKET_PRICE_DEVELOPMENT_PER_MONTH MPD_20 ON MPD_20.HOTEL_CITY_ID  = BKG.HOTEL_CITY_ID AND MPD_20.MPD_BASE_YEAR = 2019 AND MPD_20.DEVELOPMENT_YEAR = 2020
GROUP BY 
BKG.HOTEL_ID,
BKG.HOTEL_COUNTRY_NAME,
BKG.CONTINENT_NAME,
BKG.HOTEL_CATEGORY_SHORT_DESC,
BKG.HOTEL_CATEGORY_NUMBER,
--SD.SOURCING_DESTINATION,
BKG.HOTEL_COUNTRY_ID,
BKG.HOTEL_CITY_ID,
BKG.HOTEL_CATEGORY_ID,
LOCAL.ARR_YEAR ,
LOCAL.ARR_MONTH,
LOCAL.BKG_YEAR,
LOCAL.BKG_MONTH,
BKG.BKG_STATUS,
LOCAL.MPD_2019,
LOCAL.MPD_2020

"""

In [3]:
sql_query1 = """WITH HOTELS_2020_AFTER_LOCKDOWN AS(
SELECT 
DISTINCT A.HOTEL_ID
FROM
(
SELECT distinct 
BKG.BOOKING_ID,
BKG.HOTEL_ID,
CO.HOTEL_COUNTRY_NAME,
CON.CONTINENT_NAME,
HOC.HOTEL_CATEGORY_SHORT_DESC,
HOC.HOTEL_CATEGORY_NUMBER,

HO.HOTEL_COUNTRY_ID,
HO.HOTEL_CITY_ID,
HO.HOTEL_CATEGORY_ID,
BKG.HOTEL_ARRIVAL_DATE,
BKG.BOOKING_DATE,
BKG.AMOUNT_TURNOVER,
BKG.NUMBER_BOOKED_ROOMNIGHTS ,
CASE WHEN BOOKING_ORIGIN_COUNTRY_ID = HO.HOTEL_COUNTRY_ID THEN 1 END AS 'DOMESTIC',
CASE WHEN  BOOKING_ORIGIN_COUNTRY_ID <> HO.HOTEL_COUNTRY_ID THEN 1 END AS 'INTERNATIONAL',
CASE WHEN BKG.BOOKING_STATUS_ID IN (0,1) THEN 'SUCCESFFUL_BKG'
ELSE 'CANCELLED_BKG' END AS 'BKG_STATUS'
FROM DWHBIL.FAK_BOOKING BKG 
LEFT JOIN DWHBIL.V_LKP_HOTEL HO ON HO.HOTEL_ID = BKG.HOTEL_ID
LEFT JOIN DWHBIL.FAK_RATEGAIN GA ON GA.HOTEL_ID = HO.HOTEL_ID
LEFT JOIN DWHBIL.V_LKP_HOTEL_CATEGORY HOC ON HOC.HOTEL_CATEGORY_ID = HO.HOTEL_CATEGORY_ID
LEFT JOIN DWHBIL.V_LKP_HOTEL_COUNTRY CO ON CO.HOTEL_COUNTRY_ID = HO.HOTEL_COUNTRY_ID
LEFT JOIN DWHBIL.V_LKP_CONTINENT CON ON CON.CONTINENT_ID = CO.HOTEL_CONTINENT_ID
JOIN DWHBIL.REL_CITY_TO_SOURCING_DESTINATION SD ON SD.HOTEL_CITY_ID = HO.HOTEL_CITY_ID
WHERE 
 BKG.MICE_ID  = -1 AND BKG.BOOKING_SOURCE_ID <> 742 
AND (YEAR(BKG.BOOKING_DATE) IN (2020) AND MONTH(BKG.BOOKING_DATE) IN (1,2))
AND (YEAR(BKG.HOTEL_ARRIVAL_DATE) IN (2020) AND MONTH(BKG.HOTEL_ARRIVAL_DATE) IN (1,2))
AND HOC.HOTEL_CATEGORY_NUMBER IN (3,4,5) 
AND BKG.NUMBER_BOOKED_ROOMNIGHTS > 0  AND BKG.AMOUNT_TURNOVER/BKG.NUMBER_BOOKED_ROOMNIGHTS BETWEEN 10 AND 500
) A 
)
, HOTELS_2019_JF AS(
SELECT distinct 
BKG.BOOKING_ID,
BKG.HOTEL_ID,
CO.HOTEL_COUNTRY_NAME,
CON.CONTINENT_NAME,
HOC.HOTEL_CATEGORY_SHORT_DESC,
HOC.HOTEL_CATEGORY_NUMBER,

HO.HOTEL_COUNTRY_ID,
HO.HOTEL_CITY_ID,
HO.HOTEL_CATEGORY_ID,
(BKG.HOTEL_ARRIVAL_DATE) ,
(BKG.BOOKING_DATE) ,
(BKG.AMOUNT_TURNOVER) ,
(BKG.NUMBER_BOOKED_ROOMNIGHTS),
CASE WHEN BOOKING_ORIGIN_COUNTRY_ID = HO.HOTEL_COUNTRY_ID THEN 1 END AS 'DOMESTIC',
CASE WHEN  BOOKING_ORIGIN_COUNTRY_ID <> HO.HOTEL_COUNTRY_ID THEN 1 END AS 'INTERNATIONAL',
CASE WHEN  BOOKING_ORIGIN_COUNTRY_ID IS NULL THEN 1 END AS 'UNKNOWNTRAV',
CASE WHEN BKG.BOOKING_STATUS_ID IN (0,1) THEN 'SUCCESFFUL_BKG'
ELSE 'CANCELLED_BKG' END AS 'BKG_STATUS'
FROM DWHBIL.FAK_BOOKING BKG 
LEFT JOIN DWHBIL.V_LKP_HOTEL HO ON HO.HOTEL_ID = BKG.HOTEL_ID
LEFT JOIN DWHBIL.FAK_RATEGAIN GA ON GA.HOTEL_ID = HO.HOTEL_ID
LEFT JOIN DWHBIL.V_LKP_HOTEL_CATEGORY HOC ON HOC.HOTEL_CATEGORY_ID = HO.HOTEL_CATEGORY_ID
LEFT JOIN DWHBIL.V_LKP_HOTEL_COUNTRY CO ON CO.HOTEL_COUNTRY_ID = HO.HOTEL_COUNTRY_ID
LEFT JOIN DWHBIL.V_LKP_CONTINENT CON ON CON.CONTINENT_ID = CO.HOTEL_CONTINENT_ID
JOIN DWHBIL.REL_CITY_TO_SOURCING_DESTINATION SD ON SD.HOTEL_CITY_ID = HO.HOTEL_CITY_ID
WHERE 
 BKG.MICE_ID  = -1 AND BKG.BOOKING_SOURCE_ID <> 742 
AND (
((YEAR(BKG.BOOKING_DATE) IN (2019) AND MONTH(BKG.BOOKING_DATE) IN (1,2))AND (YEAR(BKG.HOTEL_ARRIVAL_DATE) IN (2019) AND MONTH(BKG.HOTEL_ARRIVAL_DATE) IN (1,2)))
OR 
((YEAR(BKG.BOOKING_DATE) IN (2019) AND MONTH(BKG.BOOKING_DATE) IN (3,4,5,6)) AND (YEAR(BKG.HOTEL_ARRIVAL_DATE) IN (2019) AND MONTH(BKG.HOTEL_ARRIVAL_DATE) IN (3,4,5,6)))
OR 
((YEAR(BKG.BOOKING_DATE) IN (2020) AND MONTH(BKG.BOOKING_DATE) IN (1,2)) AND (YEAR(BKG.HOTEL_ARRIVAL_DATE) IN (2020) AND MONTH(BKG.HOTEL_ARRIVAL_DATE) IN (1,2)))
OR 
((YEAR(BKG.BOOKING_DATE) IN (2020) AND MONTH(BKG.BOOKING_DATE) IN (3,4,5,6)) AND (YEAR(BKG.HOTEL_ARRIVAL_DATE) IN (2020) AND MONTH(BKG.HOTEL_ARRIVAL_DATE) IN (3,4,5,6)))
)
AND HOC.HOTEL_CATEGORY_NUMBER IN (3,4,5) 
AND BKG.NUMBER_BOOKED_ROOMNIGHTS > 0  AND BKG.AMOUNT_TURNOVER / BKG.NUMBER_BOOKED_ROOMNIGHTS BETWEEN 10 AND 500
AND HO.HOTEL_COUNTRY_ID = 29
AND HO.HOTEL_ID IN (SELECT distinct HOTEL_ID FROM HOTELS_2020_AFTER_LOCKDOWN)
)
SELECT DISTINCT 
COUNT(BKG.BOOKING_ID) AS BOOKING_COUNT,
BKG.HOTEL_ID,
BKG.HOTEL_COUNTRY_NAME,
BKG.CONTINENT_NAME,
BKG.HOTEL_CATEGORY_SHORT_DESC,
BKG.HOTEL_CATEGORY_NUMBER,

BKG.HOTEL_COUNTRY_ID,
BKG.HOTEL_CITY_ID,
BKG.HOTEL_CATEGORY_ID,
YEAR(BKG.HOTEL_ARRIVAL_DATE) AS ARR_YEAR ,
MONTH(BKG.HOTEL_ARRIVAL_DATE) AS ARR_MONTH,
YEAR(BKG.BOOKING_DATE) AS BKG_YEAR,
MONTH(BKG.BOOKING_DATE) AS BKG_MONTH,
SUM(BKG.AMOUNT_TURNOVER) AS TOVER,
SUM(BKG.NUMBER_BOOKED_ROOMNIGHTS) AS RNS,
SUM(DOMESTIC) AS DOM_TRAV,
SUM(INTERNATIONAL) AS INT_TRAV,
SUM(UNKNOWNTRAV) AS UNKNOWN_TRAV,
BKG.BKG_STATUS,
LOCAL.TOVER / LOCAL.RNS AS ADR,
MPD_19.MPD_FOR_DEVELOPMENT_YEAR AS MPD_2019,
MPD_20.MPD_FOR_DEVELOPMENT_YEAR AS MPD_2020
FROM HOTELS_2019_JF BKG 
JOIN DWHBIL.REF_MARKET_PRICE_DEVELOPMENT_PER_MONTH MPD_19 ON MPD_19.HOTEL_CITY_ID  = BKG.HOTEL_CITY_ID AND MPD_19.MPD_BASE_YEAR = 2018 AND MPD_19.DEVELOPMENT_YEAR = 2019
JOIN DWHBIL.REF_MARKET_PRICE_DEVELOPMENT_PER_MONTH MPD_20 ON MPD_20.HOTEL_CITY_ID  = BKG.HOTEL_CITY_ID AND MPD_20.MPD_BASE_YEAR = 2019 AND MPD_20.DEVELOPMENT_YEAR = 2020
GROUP BY 
BKG.HOTEL_ID,
BKG.HOTEL_COUNTRY_NAME,
BKG.CONTINENT_NAME,
BKG.HOTEL_CATEGORY_SHORT_DESC,
BKG.HOTEL_CATEGORY_NUMBER,

BKG.HOTEL_COUNTRY_ID,
BKG.HOTEL_CITY_ID,
BKG.HOTEL_CATEGORY_ID,
LOCAL.ARR_YEAR ,
LOCAL.ARR_MONTH,
LOCAL.BKG_YEAR,
LOCAL.BKG_MONTH,
BKG.BKG_STATUS,
LOCAL.MPD_2019,
LOCAL.MPD_2020"""

In [4]:
df = connect.export_to_pandas(sql_query)
df1 = connect.export_to_pandas(sql_query1)

In [5]:
df.info()

In [83]:
df1.info()

In [96]:
output = df.append(df1, ignore_index=True)

In [97]:
output.info()

### Country share of data

In [155]:
output['HOTEL_COUNTRY_NAME'].value_counts()[:20]

In [98]:
display(output['HOTEL_COUNTRY_NAME'].value_counts(normalize = True)[:20].sort_values(ascending=False))

### Continent data distribution

In [99]:
%matplotlib inline
output['CONTINENT_NAME'].value_counts(normalize=True).plot(kind='bar')

### Country data distribution

In [100]:
output['HOTEL_COUNTRY_NAME'].value_counts(normalize=True)[:20].plot(kind='bar')

#### To dataframe and df to dict

In [101]:
dist = output['HOTEL_COUNTRY_NAME'].value_counts(normalize = True).sort_values(ascending=False)
#dist_dict = dist.set_index('HOTEL_COUNTRY_NAME')['value'].to_dict()
dist = dist.to_frame()
dist = dist.reset_index()
dist.columns = ['country_name', 'data_share']
dist_dic = dict(zip(dist.country_name, dist.data_share))
dist_dic

#### Merging the distribution

In [118]:
#dist = dist.reset_index()
#output = output.reset_index()
# output = output.merge(dist, left_on='HOTEL_COUNTRY_NAME', right_on='country_name')
display(output.info())

#### HTML display

In [111]:
from IPython.display import display, HTML
display(HTML(output.drop_duplicates(['HOTEL_COUNTRY_NAME', 'data_share'])[['HOTEL_COUNTRY_NAME', 'data_share']].sort_values(by=['data_share'], ascending=False).to_html(index=False)))

In [153]:
#out_group = output.groupby(['HOTEL_COUNTRY_NAME','ARR_YEAR','ARR_MONTH','BKG_YEAR','BKG_MONTH'])["BOOKING_COUNT", "RNS", "DOM_TRAV", "INT_TRAV", "UNKNOWN_TRAV"].sum()
#out_group.info()
# out_group.iloc[out_group.index.get_level_values('HOTEL_COUNTRY_NAME') == "Germany"]
out_group.query('HOTEL_COUNTRY_NAME == "Germany"')

In [142]:
test = output.groupby('HOTEL_COUNTRY_NAME')
test.get_group("Germany")